# Beyond Attention: Linear Attention & Mamba SSMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/linear_attention_ssm.ipynb)

Softmax attention costs `O(n²)` in sequence length. This notebook builds two linear-time
alternatives from scratch in PyTorch, benchmarks them against softmax attention, and shows
why *selectivity* (the Mamba idea) matters:

1. **Softmax attention** — the quadratic baseline that materialises an `n × n` matrix.
2. **Linear attention** — the associativity trick, and its equivalence to a linear RNN.
3. **Diagonal state-space models (S4 → Mamba)** — the recurrent/convolutional duality and
   input-dependent selectivity.

Companion post: *Beyond Attention: Linear Attention & Mamba SSMs* on sesen.ai.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(4)
print("torch", torch.__version__)

## 1. Softmax attention: the quadratic baseline

The `scores` tensor is `(B, H, T, T)`: every query dotted with every key. That `n × n`
matrix is the whole quadratic cost, and the softmax sits in the middle, so you cannot
avoid building it.

In [ ]:
def softmax_attention_causal(q, k, v):
    """Standard causal attention. q,k,v: (B, H, T, d). O(T^2) time and memory."""
    d = q.shape[-1]
    scores = q @ k.transpose(-1, -2) / math.sqrt(d)          # (B, H, T, T)  <-- quadratic
    T = q.shape[-2]
    causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=q.device))
    scores = scores.masked_fill(~causal, float("-inf"))
    return scores.softmax(dim=-1) @ v

## 2. Linear attention: drop the softmax, reorder the matmul

Replace `softmax(QKᵀ)V` with a positive feature map `φ` applied to `Q` and `K` separately.
With the softmax gone, matrix multiplication is associative, so you can compute `φ(K)ᵀ V`
first. That product is `d × d`: its size depends on the head dimension, not the sequence
length. A running state `S` carries it across chunks, so nothing of size `T × T` is built.

In [ ]:
def feature_map(x):
    """Positive feature map phi(x) = elu(x) + 1 (Katharopoulos et al., 2020)."""
    return F.elu(x) + 1.0

def linear_attention_causal(q, k, v, chunk=128, eps=1e-6):
    """Causal linear attention via a chunked scan. q,k,v: (B, H, T, d). O(T)."""
    B, H, T, d = q.shape
    qf, kf = feature_map(q), feature_map(k)
    out = torch.empty_like(v)
    S = torch.zeros(B, H, d, v.shape[-1], device=q.device)   # KV state (d x d)
    z = torch.zeros(B, H, d, device=q.device)                # normaliser state
    for s in range(0, T, chunk):
        e = min(s + chunk, T)
        qc, kc, vc = qf[:, :, s:e], kf[:, :, s:e], v[:, :, s:e]
        att = qc @ kc.transpose(-1, -2)
        att = att.masked_fill(~torch.tril(torch.ones(e - s, e - s,
              dtype=torch.bool, device=q.device)), 0.0)      # causal within chunk
        num = att @ vc + qc @ S                              # intra + carried state
        den = att.sum(-1, keepdim=True) + (qc * z.unsqueeze(2)).sum(-1, keepdim=True)
        out[:, :, s:e] = num / (den + eps)
        S = S + kc.transpose(-1, -2) @ vc                    # advance the state
        z = z + kc.sum(dim=2)
    return out

## 3. Diagonal state-space model (S4 → Mamba)

A state-space model runs a linear recurrence `h_t = A h_{t-1} + B x_t` and reads out
`y_t = C h_t`. When `A, B, C` are fixed the layer is time-invariant and equals a
convolution, which we can compute with an FFT (the S4 training view).

In [ ]:
def ssm_conv_lti(x, A, B, C):
    """LTI diagonal SSM as a causal FFT convolution. x: (batch, T, D). A,B,C: (D, N)."""
    batch, T, D = x.shape
    L = T
    powers = torch.arange(L).view(1, L, 1)
    kernel = (C.unsqueeze(1) * (A.unsqueeze(1) ** powers) * B.unsqueeze(1)).sum(-1)  # (D, L)
    n = 1
    while n < T + L:
        n *= 2
    Xf = torch.fft.rfft(x.transpose(1, 2), n=n)
    Kf = torch.fft.rfft(kernel, n=n)
    y = torch.fft.irfft(Xf * Kf.unsqueeze(0), n=n)[..., :T]
    return y.transpose(1, 2)

### The cost crossover

Time all three across growing sequence lengths. Softmax fits a log-log slope near 2
(quadratic); linear attention and the SSM sit near slope 1 (linear).

In [ ]:
def bench(fn, *a, warmup=1, iters=5):
    for _ in range(warmup):
        fn(*a)
    best = float("inf")
    for _ in range(iters):
        t0 = time.perf_counter(); fn(*a); best = min(best, time.perf_counter() - t0)
    return best * 1e3  # ms

H, d, N, Dm = 8, 64, 16, 512
lengths = [128, 256, 512, 1024, 2048, 4096]
A = torch.rand(Dm, N) * 0.4 + 0.5
Bp, Cp = torch.randn(Dm, N) * 0.1, torch.randn(Dm, N) * 0.1
t_soft, t_lin, t_ssm = [], [], []
for T in lengths:
    q, k, v = (torch.randn(1, H, T, d) for _ in range(3))
    x = torch.randn(1, T, Dm)
    t_soft.append(bench(softmax_attention_causal, q, k, v))
    t_lin.append(bench(linear_attention_causal, q, k, v))
    t_ssm.append(bench(lambda x: ssm_conv_lti(x, A, Bp, Cp), x))
    print(f"T={T:>5}  softmax {t_soft[-1]:7.1f} ms   linear {t_lin[-1]:6.1f} ms"
          f"   ssm {t_ssm[-1]:6.1f} ms")

for name, ys in [("softmax", t_soft), ("linear", t_lin), ("ssm", t_ssm)]:
    print(f"empirical slope {name:>8}: {np.polyfit(np.log(lengths), np.log(ys), 1)[0]:.2f}")

plt.figure(figsize=(7, 5))
plt.plot(lengths, t_soft, "o-", label="softmax attention")
plt.plot(lengths, t_lin, "s-", label="linear attention")
plt.plot(lengths, t_ssm, "^-", label="diagonal SSM")
plt.xscale("log", base=2); plt.yscale("log")
plt.xlabel("sequence length"); plt.ylabel("forward pass (ms)")
plt.title("The quadratic curve pulls away"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 4. Linear attention is a linear RNN

Write linear attention out token by token and a recurrent network appears: one state
update per token, constant memory. Use the parallel form for training, the recurrent form
for constant-memory generation. The two agree to numerical precision.

In [ ]:
def linear_attention_recurrent(q, k, v):
    """The same computation as an explicit RNN: one state update per token."""
    B, H, T, d = q.shape
    qf, kf = feature_map(q), feature_map(k)
    S = torch.zeros(B, H, d, v.shape[-1])
    z = torch.zeros(B, H, d)
    outs = []
    for t in range(T):
        S = S + kf[:, :, t].unsqueeze(-1) * v[:, :, t].unsqueeze(-2)   # outer product
        z = z + kf[:, :, t]
        num = (qf[:, :, t].unsqueeze(-1) * S).sum(-2)
        outs.append(num / ((qf[:, :, t] * z).sum(-1, keepdim=True) + 1e-6))
    return torch.stack(outs, dim=2)

q, k, v = (torch.randn(1, 2, 200, 24) for _ in range(3))
err = (linear_attention_causal(q, k, v, chunk=64) - linear_attention_recurrent(q, k, v)).abs().max()
print(f"parallel vs recurrent max abs err = {err:.2e}")

## 5. Selectivity: the Mamba idea

A time-invariant SSM processes every token with the same dynamics, so it cannot decide,
based on what it just read, to store this token and skip the next. Mamba's one change:
make `B`, `C` and the step size `Δ` functions of the input. Set `selective=False` to get a
classical LTI SSM; set it `True` for the Mamba-style selective version.

In [ ]:
class DiagonalSSM(nn.Module):
    def __init__(self, d_model, d_state=16, selective=True):
        super().__init__()
        self.d_state, self.selective = d_state, selective
        self.A_log = nn.Parameter(torch.log(torch.rand(d_model, d_state) * 0.5 + 0.5))
        self.D = nn.Parameter(torch.zeros(d_model))          # skip connection
        if selective:
            self.x_proj = nn.Linear(d_model, 2 * d_state + 1)   # -> B_t, C_t, delta_t
        else:
            self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
            self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
            self.delta = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):                                    # x: (batch, T, d_model)
        B_, T, D = x.shape
        A = -torch.exp(self.A_log)                           # diagonal, negative
        if self.selective:
            proj = self.x_proj(x)
            Bm, Cm = proj[..., :self.d_state], proj[..., self.d_state:2 * self.d_state]
            delta = F.softplus(proj[..., -1:])
            dA = torch.exp(delta.unsqueeze(-1) * A)           # input-dependent decay
            dBx = (delta.unsqueeze(-1) * Bm.unsqueeze(2)) * x.unsqueeze(-1)
            h = torch.zeros(B_, D, self.d_state)
            ys = []
            for t in range(T):
                h = dA[:, t] * h + dBx[:, t]                  # selective state update
                ys.append((Cm[:, t].unsqueeze(1) * h).sum(-1))
            y = torch.stack(ys, dim=1)
        else:
            delta = F.softplus(self.delta)
            dA, dB = torch.exp(delta.unsqueeze(-1) * A), delta.unsqueeze(-1) * self.B
            h = torch.zeros(B_, D, self.d_state)
            ys = []
            for t in range(T):
                h = dA * h + dB * x[:, t].unsqueeze(-1)       # fixed dynamics
                ys.append((self.C * h).sum(-1))
            y = torch.stack(ys, dim=1)
        return y + x * self.D

class SSMClassifier(nn.Module):
    def __init__(self, vocab, d_model=64, d_state=16, n_layers=2, selective=True):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([DiagonalSSM(d_model, d_state, selective)
                                     for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.head = nn.Linear(d_model, vocab)

    def forward(self, tokens):
        h = self.embed(tokens)
        for block, norm in zip(self.blocks, self.norms):
            h = h + block(norm(h))
        return self.head(h[:, -1])                           # logits at last position

### Associative recall: content-based lookup

Show the model key-value pairs, then a query key, and ask for the matching value.
Answering needs content-based lookup, the exact thing a fixed convolution cannot do.
Train the selective and non-selective variants side by side.

In [ ]:
def make_batch(batch, n_pairs=4, n_keys=8, n_vals=8):
    """[k1,v1,...,kK,vK, query_key] -> value bound to query_key."""
    vocab = 1 + n_keys + n_vals
    seqs = np.zeros((batch, 2 * n_pairs + 1), dtype=np.int64)
    tgt = np.zeros(batch, dtype=np.int64)
    for b in range(batch):
        keys = np.random.choice(np.arange(1, 1 + n_keys), size=n_pairs, replace=False)
        vals = np.random.randint(1 + n_keys, 1 + n_keys + n_vals, size=n_pairs)
        seqs[b, 0:2 * n_pairs:2] = keys
        seqs[b, 1:2 * n_pairs:2] = vals
        j = np.random.randint(n_pairs)
        seqs[b, -1] = keys[j]; tgt[b] = vals[j]
    return torch.from_numpy(seqs), torch.from_numpy(tgt), vocab

def train_eval(selective, steps=600, batch=128, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    _, _, vocab = make_batch(1)
    model = SSMClassifier(vocab, 64, 16, n_layers=2, selective=selective)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    lossf = nn.CrossEntropyLoss()
    for _ in range(steps):
        x, y, _ = make_batch(batch)
        opt.zero_grad(); lossf(model(x), y).backward(); opt.step()
    xe, ye, _ = make_batch(2000)
    with torch.no_grad():
        return (model(xe).argmax(-1) == ye).float().mean().item()

acc_sel = train_eval(selective=True)
acc_lti = train_eval(selective=False)
print(f"selective SSM (Mamba-style): {acc_sel*100:5.1f}%")
print(f"non-selective SSM (LTI/S4) : {acc_lti*100:5.1f}%   (chance = 12.5%)")

## Exercises

1. **State size.** Sweep `d_state` in `{2, 4, 8, 16, 32}` for the selective SSM. How small
   can the state be and still solve recall?
2. **Harder recall.** Raise `n_pairs` and `n_keys`. Where does the selective model start to
   fail, and does more state or more layers help?
3. **Hybrid.** Replace one SSM block with a softmax-attention block. Does the hybrid learn
   faster than either pure model?
4. **Feature maps.** Swap `elu(x)+1` for other positive maps (for example `F.relu`, or a
   random-feature map) in linear attention and compare outputs against softmax attention.
5. **Selective scan speed.** The selective SSM here uses a Python `for` loop. Rewrite the
   scan with `torch.cumsum` tricks or `associative_scan` and re-time it.

### References

- Katharopoulos et al. (2020), *Transformers are RNNs*, https://arxiv.org/abs/2006.16236
- Gu, Goel & Ré (2022), *S4: Efficiently Modeling Long Sequences*, https://arxiv.org/abs/2111.00396
- Gu & Dao (2023), *Mamba: Selective State Spaces*, https://arxiv.org/abs/2312.00752
- Yang et al. (2024), *Gated Linear Attention*, https://arxiv.org/abs/2312.06635